# 3. Learned ratios and unbinned Asimov fits


    Train process-to-S and systematic-variation density ratios with the pinned
    NSBI toolkit, construct the interference model, and compare learned and
    analytical unbinned Asimov scans. Run notebooks 1–2 first.

    Here NSBI ratio estimation uses classifiers trained with cross entropy; it
    does not require a separate normalizing-flow density estimator because the
    S reference can be sampled directly.

### Runtime and persistent run

In Colab, choose **Runtime → Change runtime type → GPU** for notebooks 2–3.
Both PyTorch training and JAX likelihood fits use the GPU when available.
`JAX_BACKEND = "auto"` detects the Colab NVIDIA GPU; choose `"gpu"` to require it,
or `"cpu"` for an explicit CPU run. Setup installs matching JAX/JAXlib and CUDA 12
plugin packages, then checks double-precision JIT computation and gradients on
the selected device before training. GPU initialization failures are reported
instead of silently switching to CPU. Restart the runtime once when switching
from the previous CPU-only setup or after a JAX plugin error.
For a runtime-only repair, keep the same run name: saved samples and trained
networks are reused. A changed physics model requires a new run, as below.
The source checkout lives in the temporary runtime; **datasets, checkpoints,
workspace files, and results live in Google Drive**. Use the same `RUN_NAME`
and `MODE` in every notebook. Set `MODE = "smoke"` for a short workflow check;
production generates five million events each for S, SBI and B, and fifty
million each for NI, NI_up and NI_down.
Use `CONFIG_OVERRIDES` to change a computational budget without editing source,
for example `{"quadrature_per_process": 500_000, "epochs": 150}`. Use the same
overrides in all notebooks, and choose a new `RUN_NAME` when changing them.

This repository may be private. Grant Colab access when opening it from GitHub.
For the runtime download, either save a GitHub fine-grained token with **Contents:
read** access to this repository as the Colab secret `GITHUB_TOKEN`, or enter it
in the hidden prompt. The token is used only in a Python HTTP header and is not
written into files, Git remotes, shell arguments, or notebook output. Alternatively,
upload a repository ZIP to `/content/poodemo_source.zip`; no token is then needed.
An existing `/content/poodemo` checkout is reused. Delete that checkout to fetch
new source after updating the repository; your Drive run is separate.
For the enlarged NI bank and wider ratio classifiers, use the new default
`RUN_NAME = "paper-distinct-s-v4"`. Rerun notebooks 01–03; then rerun 04–09
to refresh their downstream results. The physics amplitudes are unchanged, but
the sample/training configuration is new (schema 7). Start a fresh Colab runtime
or refresh the temporary source checkout and restart it before proceeding.
Use the same new run name in every notebook; previous Drive runs remain intact.

In [ ]:
import os
import sys
from pathlib import Path

RUN_NAME = "paper-distinct-s-v4"
MODE = os.environ.get("POODEMO_MODE", "production")  # production or smoke
CONFIG_OVERRIDES = {}  # e.g. {"quadrature_per_process": 500_000, "epochs": 150}
JAX_BACKEND = os.environ.get("POODEMO_JAX_BACKEND", "auto")  # auto, gpu, or cpu

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    requested_backend = JAX_BACKEND.strip().lower()
    if requested_backend not in ("auto", "gpu", "cpu"):
        raise ValueError("JAX_BACKEND must be auto, gpu, or cpu")
    selected_backend = requested_backend
    if requested_backend == "auto":
        try:
            gpu_probe = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                                       capture_output=True, text=True, timeout=10)
            gpu_present = gpu_probe.returncode == 0 and bool(gpu_probe.stdout.strip())
        except (OSError, subprocess.TimeoutExpired):
            gpu_present = False
        selected_backend = "gpu" if gpu_present else "cpu"
    # CUDA-only selection fails visibly if initialization fails. Keep PyTorch's
    # CUDA visibility and avoid JAX reserving most GPU memory before training.
    jax_platform = "cuda" if selected_backend == "gpu" else "cpu"
    os.environ["JAX_PLATFORMS"] = jax_platform
    os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT = Path(os.environ.get("POODEMO_ROOT", f"/content/drive/MyDrive/poodemo/runs/{RUN_NAME}"))
    REPO_DIR = Path("/content/poodemo")
    if not (REPO_DIR / "pyproject.toml").exists():
        import getpass
        import io
        import shutil
        import tarfile
        import tempfile
        import urllib.error
        import urllib.request
        import zipfile

        uploaded_zip = Path("/content/poodemo_source.zip")
        if uploaded_zip.exists():
            archive_bytes = uploaded_zip.read_bytes()
            archive_kind = "zip"
        else:
            archive_url = "https://api.github.com/repos/rafaellopesdesa/poodemo/tarball/main"

            def fetch_source(token=None):
                headers = {"Accept": "application/vnd.github+json", "User-Agent": "poodemo-colab"}
                if token:
                    headers["Authorization"] = "Bearer " + token
                request = urllib.request.Request(archive_url, headers=headers)
                with urllib.request.urlopen(request, timeout=180) as response:
                    return response.read()

            try:
                archive_bytes = fetch_source()
            except urllib.error.HTTPError as error:
                if error.code not in (401, 403, 404):
                    raise RuntimeError(f"Repository download failed (HTTP {error.code}).") from None
                token = None
                try:
                    from google.colab import userdata
                    token = userdata.get("GITHUB_TOKEN")
                except Exception:
                    pass
                if not token:
                    token = getpass.getpass("GitHub token with read access to poodemo: ").strip()
                if not token:
                    raise RuntimeError("Upload /content/poodemo_source.zip or provide read access.")
                try:
                    archive_bytes = fetch_source(token)
                except urllib.error.HTTPError as auth_error:
                    raise RuntimeError(f"Repository download failed (HTTP {auth_error.code}); check token access.") from None
                finally:
                    token = None
            archive_kind = "tar"

        # Validate archive paths and reject links before extracting the source.
        with tempfile.TemporaryDirectory(prefix="poodemo-source-") as temp:
            staging = Path(temp).resolve()

            def safe_destination(name):
                destination = (staging / name).resolve()
                if not destination.is_relative_to(staging):
                    raise RuntimeError("Unsafe path in source archive.")
                return destination

            if archive_kind == "zip":
                with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
                    for entry in archive.infolist():
                        safe_destination(entry.filename)
                        if (entry.external_attr >> 16) & 0o170000 == 0o120000:
                            raise RuntimeError("Symbolic links are not supported in the source ZIP.")
                    archive.extractall(staging)
            else:
                with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode="r:gz") as archive:
                    for entry in archive.getmembers():
                        safe_destination(entry.name)
                        if not (entry.isfile() or entry.isdir()):
                            raise RuntimeError("Links and special files are not supported in the source archive.")
                    archive.extractall(staging)
            candidates = [p.parent for p in staging.rglob("pyproject.toml") if (p.parent / "poodemo").is_dir()]
            if len(candidates) != 1:
                raise RuntimeError("The uploaded archive must contain one poodemo repository checkout.")
            shutil.copytree(candidates[0], REPO_DIR, dirs_exist_ok=True)
        del archive_bytes

    if os.environ.get("POODEMO_SKIP_INSTALL") != "1":
        from importlib.metadata import version, PackageNotFoundError
        # JAX discovers every installed plugin, even when selecting CPU.
        # Retain already-matched CUDA 12 packages when rerunning GPU setup.
        remove_plugins = []
        for package in ("jax-cuda12-plugin", "jax-cuda12-pjrt", "jax-cuda13-plugin", "jax-cuda13-pjrt"):
            try:
                installed_version = version(package)
            except PackageNotFoundError:
                continue
            if not (selected_backend == "gpu" and package.startswith("jax-cuda12-") and installed_version == "0.5.3"):
                remove_plugins.append(package)
        if remove_plugins:
            subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", *remove_plugins])
        gpu_requirements = (["jax==0.5.3", "jaxlib==0.5.3",
                             "jax-cuda12-plugin[with-cuda]==0.5.3", "jax-cuda12-pjrt==0.5.3"]
                            if selected_backend == "gpu" else [])
        # Resolve with the base requirements so PyTorch's CUDA dependencies are
        # considered together with JAX's. Avoid blanket --upgrade changes.
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r",
                               str(REPO_DIR / "requirements-colab.txt"), *gpu_requirements])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])

        # Check the live notebook kernel before any expensive network training.
        import jax
        import jaxlib
        import jax.numpy as jnp
        if jax.__version__ != "0.5.3" or jaxlib.__version__ != "0.5.3":
            raise RuntimeError("JAX packages are stale in this kernel. Restart the runtime and rerun setup.")
        if selected_backend == "gpu":
            for package in ("jax-cuda12-plugin", "jax-cuda12-pjrt"):
                if version(package) != "0.5.3":
                    raise RuntimeError(f"{package} must match JAX 0.5.3. Restart the runtime and rerun setup.")
        jax.config.update("jax_platforms", jax_platform)
        jax.config.update("jax_enable_x64", True)
        try:
            jax_devices = jax.devices()
            if not jax_devices or any(device.platform != selected_backend for device in jax_devices):
                raise RuntimeError(f"Expected {selected_backend} devices; found {jax_devices}")
            probe_value, probe_gradient = jax.jit(jax.value_and_grad(lambda x: jnp.sum(x*x)))(
                jnp.asarray([1., 2., 3.], dtype=jnp.float64))
            probe_gradient.block_until_ready()
            if (float(probe_value) != 14. or list(probe_gradient) != [2., 4., 6.]
                    or probe_gradient.dtype != jnp.float64
                    or any(device.platform != selected_backend for device in probe_gradient.devices())):
                raise RuntimeError("JAX double-precision value/gradient check failed")
        except Exception as error:
            raise RuntimeError(
                f"JAX {selected_backend} startup failed. Select a Colab GPU runtime if requesting GPU, "
                "then restart the runtime and rerun setup. Use JAX_BACKEND='cpu' only for an intentional CPU run."
            ) from error
        print(f"JAX {jax.__version__} / jaxlib {jaxlib.__version__}: backend={selected_backend}, {jax_devices}; float64 JIT/gradient OK")

    # A new editable install is not activated in an already-running kernel.
    # Expose the inner package directly, including after a failed import cached
    # the outer /content/poodemo checkout as a namespace package.
    import importlib
    source_root = str(REPO_DIR.resolve())
    if source_root in sys.path:
        sys.path.remove(source_root)
    sys.path.insert(0, source_root)
    importlib.invalidate_caches()
    cached_package = sys.modules.get("poodemo")
    cached_file = getattr(cached_package, "__file__", None)
    expected_init = REPO_DIR / "poodemo" / "__init__.py"
    if cached_package is not None and (cached_file is None or Path(cached_file).resolve() != expected_init.resolve()):
        for module_name in list(sys.modules):
            if module_name == "poodemo" or module_name.startswith("poodemo."):
                del sys.modules[module_name]
else:
    # Locally: pip install -r requirements-colab.txt && pip install -e .
    ROOT = Path(os.environ.get("POODEMO_ROOT", str(Path.home() / "poodemo-runs" / RUN_NAME)))
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "poodemo" / "pipeline.py").exists():
            sys.path.insert(0, str(candidate))
            break

ROOT.mkdir(parents=True, exist_ok=True)
print(f"Run directory: {ROOT}")
print(f"Mode: {MODE}")

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from poodemo.pipeline import create_run

run = create_run(ROOT, mode=MODE, overrides=CONFIG_OVERRIDES)
plt.rcParams.update({"figure.figsize": (7, 4.5), "axes.grid": True, "grid.alpha": 0.2})
display(pd.Series(run.config, name="configuration"))

### Density ratios and the physical model

For the fixed selected phase space, set \(q=p_S^{\rm sel}\), and learn
\(r_j=p_j^{\rm sel}/q\) for the other processes. Shape ratios and physical
yields are distinct. At the nominal nuisance value \(\alpha_{NI}=0\),
\[
\frac{D(x;\mu)}{q(x)}
=(\mu-\sqrt\mu)\lambda_S
 +\sqrt\mu\lambda_{SBI}r_{SBI}(x)
 +(1-\sqrt\mu)\lambda_Br_B(x)
 +\lambda_{NI}r_{NI}(x),
\]
where every \(\lambda\) is now a **selected** yield.

We also train the NI up/down variation-to-nominal ratios. There are five
ratio tasks: SBI/S, B/S, NI/S, NI_up/NI, and NI_down/NI. The production
three-member ensembles therefore contain fifteen ratio networks, plus
the separate preselection network. Only NI is morphed; S, B, and SBI
retain their nominal templates. The workspace parameters are
\((\mu,\alpha_{NI})\).

The nuisance interpolation is the toolkit's exponential–polynomial
interpolation. A standard-normal auxiliary measurement supplies
\[
L_{\rm aux}(\alpha_{NI})\propto\exp(-\alpha_{NI}^2/2),
\]
so its contribution to \(-2\log L\) is \(+\alpha_{NI}^2\).
This constraint is included in every profiled likelihood. The
interpolated nuisance model need not equal a continuously shifted
Gaussian amplitude away from its nominal and ±1 anchors.

### Classifier architecture and training metadata

Each production ratio classifier has **3 inputs → 1024 → 1024 → 1024 →
1 logit → Sigmoid**, with a bias in each linear layer and **SiLU/Swish**,
$a\mapsto a\,\sigma(a)$, after each hidden layer. The public network output is
$s=\sigma(\ell)\in[0,1]$. Standard mean **binary cross entropy** is computed
using its numerically stable BCE-with-logits form on the pre-sigmoid logit.
This is mathematically the same sigmoid+BCE objective. It avoids saturation
when evaluating the loss and $r=s/(1-s)=\exp(\ell)$ in the tails; sigmoid
itself is not an additional calibration correction.

Production uses batch size **1024**, labels 0=denominator and 1=numerator,
and equal class minibatches. There is **no dropout, batch normalization,
weight decay, L1/L2 penalty, or label smoothing**. NAdam starts at $10^{-3}$;
the learning rate decreases geometrically to **$10^{-9}$ at epoch 91**
(1-based) and stays there through epoch 100. Ratio early stopping is disabled
so the final low-rate epochs really run; the best-validation checkpoint is
still retained. The used rate in every epoch is recorded in `history.json`.
Preselection retains its separate original network settings.

The table below reads the actual loaded layers, activations and parameter
counts; smoke/custom runs can differ from these production dimensions.
The three production members are combined by averaging ratios, not logits.
Mean normalization on separate calibration events is a later operation, not
a loss penalty or a guarantee of shape closure.

This configuration requires the new v4 run. Matching completed v4 checkpoints
can be reused. Per-task/per-member metadata is saved in each `training.json`
and in `results/03_architecture.json`.

### Independent density validation bank

These diagnostics generate **new events**, apply the frozen selector from
notebook 02, and are prepared **before ratio training**. The bank is separate
from training, checkpoint-selection validation, calibration and workspace normalization.
The production default is two million **generated** events per source
(S, SBI, B, NI and NI up/down); selected counts are printed below. Generation
is cached with the physics/selection identity and random seed. Re-running with
the same settings reuses it. These controls are diagnostic budgets and do not
change `RUN_NAME`, the model, or `CONFIG_OVERRIDES`.

Each reweighting figure compares numerator events with denominator events
weighted by the learned ratio, for all three coordinates. Lower panels show
reweighted/target density with Monte Carlo uncertainty. The full sample
counts define normalization; the displayed range is never renormalized to
force agreement. Bins with too few target or reference events are flagged, not
reported as successful closure.

Calibration figures compare the mean predicted score $s=r/(1+r)$ to the
observed numerator fraction using an equal class prior, with residual
$f_{\rm numerator}-\langle s\rangle$ underneath. Immediately after **each
member** finishes, its raw reweighting and calibration plots are displayed,
before training the next member. After each task, the final ratio is shown in
both raw and calibration-mean-normalized forms. Member checks do not alter
the ensemble weights or fit any calibration map. Calibration
here is a diagnostic: these cells do not fit a new calibration map. Error bars
describe the independent finite validation samples conditional on the trained
network; they do not include training or calibration uncertainty.

In [ ]:
from poodemo.closure_validation import prepare_validation_bank

DIAGNOSTIC_N_PER_SAMPLE = 20_000 if MODE == "smoke" else 2_000_000
DIAGNOSTIC_EDGES = np.linspace(-5., 6., 56)
CALIBRATION_BINS = 20
VALIDATION_LOG_SCALE = False  # Set True for logarithmic marginal-density panels.

validation_bank = prepare_validation_bank(run, n_per_sample=DIAGNOSTIC_N_PER_SAMPLE)
display(pd.DataFrame([
    {"sample": name, "generated": validation_bank["generated_counts"][name],
     "selected": len(sample)}
    for name, sample in validation_bank["samples"].items()
]))

In [ ]:
from poodemo.pipeline import train_ratios

def show_ratio_validation(report):
    print(f"{report['ratio']} — {report['stage']}", flush=True)
    for figure in report["figures"].values():
        display(figure)
    # Canvases are saved and closed by the pipeline after this callback returns.
    if report["stage"] == "ensemble":
        display(report["reweighting"].groupby("stage", sort=False)[
            ["denominator_mean_ratio", "numerator_events", "denominator_events"]].first())

ratio_diagnostics = train_ratios(
    run, validation_samples=validation_bank["samples"],
    on_validation=show_ratio_validation,
    validation_options={"edges": DIAGNOSTIC_EDGES, "score_bins": CALIBRATION_BINS,
                        "log": VALIDATION_LOG_SCALE},
)
display(ratio_diagnostics)
print("Saved per-member and ensemble plots:", run.path("plots", "03_validation"))

In [ ]:
from poodemo.pipeline import RATIO_TASKS
from poodemo.training import load_predictor, training_metadata
from poodemo.data import save_json

architecture_reports, architecture_rows, layer_rows = {}, [], []
for numerator, denominator in RATIO_TASKS:
    task = f"{numerator}_over_{denominator}"
    predictor = load_predictor(run.path("models", "ratios", task))
    report = training_metadata(predictor)
    architecture_reports[task] = report
    members = report.get("members", [report])
    for member_index, member in enumerate(members):
        architecture = member["architecture"]
        optimizer = member["optimizer"]
        architecture_rows.append({
            "ratio": f"{numerator}/{denominator}", "member": member_index,
            "inputs": architecture["input_features"],
            "hidden_layers": architecture["hidden_layers"],
            "width": architecture["hidden_width"],
            "activation": architecture["hidden_activation"],
            "output": architecture["output_activation"],
            "parameters": architecture["trainable_parameters"],
            "loss": member["loss"]["name"], "optimizer": optimizer["type"],
            "batch_size": member.get("config", {}).get("batch_size"),
            "initial_lr": optimizer["defaults"]["lr"],
            "lowest_lr_used": member.get("lowest_learning_rate_used"),
            "final_lr_used": member.get("final_learning_rate_used"),
            "epochs_completed": member.get("epochs_completed"),
            "dropout": member["regularization"]["dropout"],
            "weight_decay": optimizer["parameter_group_weight_decay"],
        })
        layer_rows.extend({"ratio": f"{numerator}/{denominator}", "member": member_index,
                           **layer} for layer in architecture["layers"])
    del predictor
save_json(run.path("results", "03_architecture.json"), architecture_reports)
display(pd.DataFrame(architecture_rows))
with pd.option_context("display.max_rows", None):
    display(pd.DataFrame(layer_rows))
print("Full optimizer, scheduler, preprocessing and checkpoint details:")
example = architecture_reports[next(iter(architecture_reports))]
example = example.get("members", [example])[0]
print(json.dumps({key: example[key] for key in
                 ("optimizer", "scheduler", "preprocessing", "checkpoint_selection")}, indent=2))

### Validate ratios before interpreting the inference

The analytical density is available in this toy. Held-out diagnostics
distinguish finite training errors from score compression and histogram
errors. Ratio normalization alone does not guarantee that the ratio is
correct point by point. Signed interference coefficients make numerical
positivity of the total learned intensity a particularly important
diagnostic; a failed physical-domain check must not be silently hidden
by clipping the final likelihood.

To isolate neural-network shape errors in this demonstration, the
learned and analytical likelihoods use the same accepted component
yields from the common analytical quadrature. Sample-estimated selected
rates are reported separately. Learned shapes are numerically normalized
on that quadrature after independent calibration, and their raw
integrals are saved. This uses the toy's known density for validation
and rate control; an analysis without an oracle would estimate those
quantities from its simulation and auxiliary measurements.

Bringing the two branches close while increasing exposure makes this a
more stringent test of learned ratios. Small shape-estimation errors
can now have a substantial likelihood effect. Agreement of the
analytical benchmark with the target landscape does not establish
neural-network closure. Inspect the held-out ratio diagnostics and
learned-versus-analytical curves; the short smoke training is only a
workflow check, and production closure must be demonstrated separately.

In [ ]:
from poodemo.pipeline import prepare_quadrature

quadrature = prepare_quadrature(run)
print('Quadrature products:', list(quadrature))

### High-statistics expectation and MLE closure

For a normalized target-to-reference density ratio, an independent denominator
sample should satisfy
\[
\widehat{E_q[r]}_N=\frac{1}{N}\sum_{i=1}^{N}r(x_i)\longrightarrow 1.
\]
The convergence plot uses increasing, nested prefixes of the selected bank,
with Monte Carlo standard errors and effective sample sizes saved in the
table. Prefix estimates are correlated. Increasing statistics can reveal a
persistent offset; it does not force an inaccurate learned ratio toward one.
No mean correction is fitted to this bank.

The unbinned likelihood diagnostic compares the analytical and learned models
on an independent, simulation-weighted expectation at **$\mu_*=1$**, and on a
separately generated high-statistics Poisson sample from SBI+NI. It shows
NI-fixed and NI-profiled scans, a vertical line at one, and fitted $\hat\mu$
values. A high-statistics expectation checks systematic MLE closure; an
individual Poisson sample also fluctuates statistically. The auxiliary NI
measurement is held at its nominal value. The learned-model self-Asimov scan
later in the notebook is a separate consistency check.

Workspace rates and shape-normalization constants remain frozen from the
original workspace quadrature. The new data never normalize the fitted model.
Therefore analytical closure itself has finite original-quadrature and new-bank
uncertainty; learned-minus-analytical differences help isolate network errors.
The Poisson-sample exposure is increased independently of the bank size and is
reported with the selected observed count. Invalid learned intensities or fits
are flagged; they are never clipped into a closing scan. Both interference
branches are included in the fit search.

These diagnostics can be expensive. Increase the controls below and the bank
budget above to examine stability. All tables and PDF/PNG figures are saved;
none of these cells retrain or recalibrate the networks.
The independent-data fits use NumPy/SciPy on CPU; the standard toolkit/JAX
scans below retain their configured backend. Both an overview and a zoom around
the fitted minima are saved, with extra scan points near each minimum.

In [ ]:
from poodemo.closure_validation import run_closure_validation

CLOSURE_SCAN_POINTS = 17 if MODE == "smoke" else 81
POISSON_GENERATED_EVENTS = 50_000 if MODE == "smoke" else 10_000_000
CLOSURE_PROFILE_NI = True

closure_validation = run_closure_validation(
    run, validation_bank, scan_points=CLOSURE_SCAN_POINTS,
    poisson_generated_events=POISSON_GENERATED_EVENTS, profile=CLOSURE_PROFILE_NI,
)
display(closure_validation["convergence"])
display(closure_validation["fits"])
display(pd.Series(closure_validation["poisson"], name="Independent Poisson sample"))
print("Closure figures, tables and provenance:", closure_validation["directory"])
for figure in closure_validation["figures"]:
    display(figure)
    plt.close(figure)

### Asimov integration and profiling

We approximate expectations with a held-out weighted integration sample.
These are Asimov likelihood curves, not a coverage study with random
Poisson pseudo-experiments. The extended likelihood retains the total
expected yield and includes the Gaussian NI auxiliary measurement.

At the configured generating point \((\mu_*,\alpha_{NI,*})=(\mu_*,0)\),
an exact intensity model gives
\[
t_A(\mu,\alpha_{NI})=2\int\!dx\,
\left[D(x;\mu,\alpha_{NI})-D_*(x)
-D_*(x)\log\frac{D(x;\mu,\alpha_{NI})}{D_*(x)}\right]
+\alpha_{NI}^2,
\]
followed by profiling. Finite quadrature and learned-ratio errors are
separate effects. The pipeline saves both analytical and learned
reference curves so that later comparisons can identify their origin.

In [ ]:
from poodemo.pipeline import run_unbinned

unbinned = run_unbinned(run)
display(unbinned['scans'])

In [ ]:
def plot_scans(frame, title, filename, group_columns=None, *, xlim=None, ylim=None):
    """Plot saved scans with optional (min, max) axis limits.

    Omit xlim/ylim to retain the default view. A None endpoint keeps that
    bound unchanged, for example ylim=(0, None) or xlim=(None, 1.2).
    """
    x_column = next((c for c in ["mu", "mu_test", "mu0"] if c in frame.columns), None)
    y_column = next((c for c in ["q", "t_mu", "test_statistic", "ts"] if c in frame.columns), None)
    if x_column is None or y_column is None:
        raise ValueError(f"Expected mu and q scan columns; found {list(frame.columns)}")
    if group_columns is None:
        group_columns = [c for c in ["method", "model", "systematics", "nuisances", "n_bins", "bins", "observable"] if c in frame.columns]
    groups = frame.groupby(group_columns, dropna=False, sort=False) if group_columns else [("scan", frame)]
    fig, ax = plt.subplots()
    for label, group in groups:
        group = group.sort_values(x_column)
        values = label if isinstance(label, tuple) else (label,)
        parts = []
        for column, value in zip(group_columns, values):
            if pd.isna(value):
                continue
            if column == "systematics":
                parts.append("profiled" if value else "stat only")
            elif column in ("n_bins", "bins"):
                parts.append(f"{int(value)} bins")
            else:
                parts.append(str(value))
        ax.plot(group[x_column], group[y_column], marker=".", ms=3, label=" | ".join(parts))
    ax.axhline(1.0, color="0.45", ls=":", lw=1)
    ax.set(xlabel=r"Tested signal strength $\mu_0$", ylabel=r"$t(\mu_0)$", title=title)
    ax.set_ylim(bottom=0)
    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.legend(fontsize=8)
    fig.tight_layout()
    output = ROOT / "plots"
    output.mkdir(exist_ok=True)
    fig.savefig(output / filename, bbox_inches="tight")
    plt.show()
    return fig

In [ ]:
SCAN_XLIM = None  # e.g. (0.02, 1.15); None keeps the default x-axis range.
SCAN_YLIM = None  # e.g. (0, 8); None keeps the default y-axis range.
plot_scans(unbinned['scans'], 'Unbinned Asimov likelihood scans', '03_unbinned_scans.pdf',
           xlim=SCAN_XLIM, ylim=SCAN_YLIM);

### Why a second minimum can appear

At nominal NI nuisance and with the selection fixed, write the selected
total yield as
\[
\Lambda_\mu=\lambda_S\mu+\lambda_I\sqrt\mu+\lambda_B+\lambda_{NI},
\qquad \lambda_I=\lambda_{SBI}-\lambda_S-\lambda_B<0.
\]
Equating this yield to the generating yield gives
\[
\Lambda_\mu-\Lambda_{\mu_*}
=(\sqrt\mu-\sqrt{\mu_*})
[\lambda_S(\sqrt\mu+\sqrt{\mu_*})+\lambda_I]=0.
\]
Besides \(\mu=\mu_*\), there is an equal-rate solution
\[
\mu_{\rm rate}=\left(-\lambda_I/\lambda_S-\sqrt{\mu_*}\right)^2
\]
when the expression in parentheses is nonnegative. This is an exact
statement about the **total rate at nominal nuisance**, not an exact
degeneracy of the full event distribution. Similar signal and
interference shapes can leave a second likelihood minimum nearby;
shape information can lift it. Profiling NI can change its depth and
location, with the Gaussian constraint penalizing the required shift.

The distinct-signal benchmark keeps the second basin below 1 rather
than forcing it near 0.5. The selected-rate turnover is
\[
\mu_{\rm turnover}=\left(-\frac{\lambda_I}{2\lambda_S}\right)^2.
\]
Neither this turnover nor the equal-rate partner is generally a
full-likelihood extremum. The analytical-selector validation gives
secondary minima near 0.120 (NI fixed) and 0.069 (NI profiled);
remeasure these locations with the trained selector below.

The following analytical-only panels separate fixed-NI and profiled-NI
curves. Dots mark interior minima on the evaluated grid, rather than
claiming a more precise optimizer location. `LANDSCAPE_Q_MAX` focuses
the vertical range on the nearby minima and intervening barrier; set
it to `None` for automatic scaling. The preceding overview figure and
saved scan table retain the full test-statistic range.

In [ ]:
from poodemo.pipeline import prepare_quadrature

LANDSCAPE_Q_MAX = 10.0  # Set None for the full vertical range.
landscape_quad = prepare_quadrature(run)
selected_s, selected_sbi, selected_b, selected_ni = landscape_quad["nominal"] @ landscape_quad["weights"]
selected_interference = selected_sbi - selected_s - selected_b
truth_mu = run.config["asimov_mu"]
other_root = -selected_interference / selected_s - np.sqrt(truth_mu)
equal_rate_mu = float(other_root**2) if other_root >= 0 else np.nan
display(pd.Series({"selected_lambda_S": selected_s,
                   "selected_lambda_I": selected_interference,
                   "generating_mu": truth_mu,
                   "other_equal_rate_mu_at_alpha_NI_0": equal_rate_mu}, name="rate diagnostic"))
analytical_scans = unbinned["scans"].query("model == 'analytic unbinned'")
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True, sharey=True)
minima = []
for ax, systematic in zip(axes, [False, True]):
    frame = analytical_scans[analytical_scans["systematics"] == systematic].sort_values("mu")
    mu_values = frame["mu"].to_numpy()
    q_values = frame["q"].to_numpy()
    ax.plot(mu_values, q_values, color="#D55E00" if systematic else "#0072B2", lw=2)
    for index in range(1, len(frame) - 1):
        nearby = q_values[index-1:index+2]
        if np.all(np.isfinite(nearby)) and nearby[1] <= min(nearby[0], nearby[2]):
            ax.scatter(mu_values[index], q_values[index], color="black", zorder=4)
            minima.append({"systematics": systematic, "mu_grid_minimum": mu_values[index], "q": q_values[index]})
    ax.axvline(truth_mu, color="0.35", ls=":", label="Generating value")
    if np.isfinite(equal_rate_mu) and run.config["mu_min"] <= equal_rate_mu <= run.config["mu_max"]:
        ax.axvline(equal_rate_mu, color="0.6", ls="--", label="Other nominal equal-rate value")
    ax.axhline(1., color="0.7", ls=":")
    ax.set(xlabel=r"Tested signal strength $\mu_0$", ylabel=r"$t_A(\mu_0)$", ylim=(0, LANDSCAPE_Q_MAX),
           title="Analytical: NI profiled" if systematic else "Analytical: NI fixed at zero")
    ax.legend(fontsize=8)
fig.tight_layout()
for extension in ("pdf", "png"):
    fig.savefig(ROOT / "plots" / f"03_analytical_landscape.{extension}", bbox_inches="tight", dpi=160)
plt.show()
display(pd.DataFrame(minima))
print("Scan range:", (run.config["mu_min"], run.config["mu_max"]))
print("Fit bounds:", run.config["mu_fit_bounds"])

### Check the numerical expectation itself

The next diagnostic evaluates the nominal-nuisance analytical Asimov
objective using the full integration sample and two disjoint halves.
The split is made **within each proposal stratum before selection**;
rejected events contribute zero to the uncertainty calculation.
Error bars estimate Monte Carlo integration uncertainty, not statistical
confidence intervals on \(\mu\). Half-sample agreement and small
estimated errors are checks on the numerical expectation; a precision
study should also increase the integration budget in a separate run.

In [ ]:
integration_check = unbinned["quadrature"].sort_values("mu")
display(integration_check)
fig, ax = plt.subplots()
ax.errorbar(integration_check["mu"], integration_check["q"],
            yerr=integration_check["standard_error"], fmt="o-", ms=3,
            capsize=2, label="Full sample ± integration SE")
ax.plot(integration_check["mu"], integration_check["q_half_even"], "--", label="Even half")
ax.plot(integration_check["mu"], integration_check["q_half_odd"], ":", label="Odd half")
ax.set(xlabel=r"Signal strength $\mu$", ylabel="Asimov likelihood objective",
       title="Integration stability: fixed nominal NI nuisance")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(ROOT / "plots" / "03_quadrature_stability.pdf", bbox_inches="tight")
plt.show()

In [ ]:
for key, value in unbinned.items():
    if key not in ("scans", "quadrature"):
        print(key)
        display(value)

The horizontal line at \(t=1\) is a visual likelihood-scale reference.
These Asimov curves alone do not demonstrate interval coverage or
establish Wilks' theorem at a boundary. The scan uses positive signal
strengths; the derivative of the \(\sqrt\mu\) model is singular at zero.
The scan range and fit bounds are printed above from the run configuration.
The positive lower fit bound is a numerical floor for
this regular interior demonstration, not an implementation of a
discovery test at the physical boundary \(\mu=0\).

Short smoke training may give visibly inaccurate learned curves or
invalid learned intensities. Such points are flagged and appear as gaps;
they do not count as successful learned-model closure. Inspect fit
validity and ratio diagnostics before increasing the training budget.

Next: **04_score_histograms.ipynb**.